# 04 — Classification
**Referencias:** ESL Cap. 4 (Linear Methods for Classification) · Géron Cap. 3, 5

## Clasificación óptima de Bayes (ESL 4.1)
El clasificador óptimo (sin estimación) asigna la clase con mayor probabilidad posterior:
$$G^*(x) = \underset{g \in \mathcal{G}}{\arg\max} \, P(G=g \mid X=x)$$

Su tasa de error es el **Bayes Error Rate** — el piso inferior de cualquier clasificador.

## Linear Discriminant Analysis (ESL 4.3)
Asume que cada clase tiene distribución Gaussiana con la **misma covarianza** $\Sigma$:
$$\delta_k(x) = x^T\Sigma^{-1}\mu_k - \frac{1}{2}\mu_k^T\Sigma^{-1}\mu_k + \log\pi_k$$

Clasifica en la clase $k$ que maximiza $\delta_k(x)$. El límite de decisión es lineal.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, classification_report
)

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

n = 2000
sessions       = np.random.randint(1, 20, n)
time_on_site   = np.random.uniform(30, 600, n)
pages          = np.random.uniform(1, 8, n)
days_since_reg = np.random.randint(0, 30, n)
channel        = np.random.choice(['organic','paid','email','direct'], n)
device         = np.random.choice(['mobile','desktop','tablet'], n)

logit = (-3 + sessions*0.15 + time_on_site*0.003 + pages*0.2
         - days_since_reg*0.05 + np.where(channel=='email',0.8,0)
         + np.where(device=='desktop',0.4,0))
prob = 1 / (1 + np.exp(-logit))
activated = (np.random.uniform(0,1,n) < prob).astype(int)

df = pd.DataFrame({
    'sessions': sessions, 'time_on_site': time_on_site.round(0),
    'pages': pages.round(2), 'days_since_reg': days_since_reg,
    'channel': channel, 'device': device, 'activated': activated,
})

num_features = ['sessions','time_on_site','pages','days_since_reg']
cat_features = ['channel','device']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
])

X = df[num_features + cat_features]
y = df['activated']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Shape: {df.shape} | Tasa activación: {y.mean():.2%}')

## 1 — LDA, QDA, Naive Bayes (ESL Cap. 4)

In [ ]:
def eval_clf(name, model):
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:,1] if hasattr(model, 'predict_proba') else None
    auc = roc_auc_score(y_test, y_prob) if y_prob is not None else float('nan')
    print(f'{name:<30} Acc:{accuracy_score(y_test,y_pred):.3f}  '
          f'F1:{f1_score(y_test,y_pred):.3f}  AUC:{auc:.3f}')
    return pipe

print(f'{"Modelo":<30} {"Acc":>6}  {"F1":>6}  {"AUC":>6}')
print('-'*55)
# Métodos de ESL Cap. 4
pipe_lda  = eval_clf('LDA (ESL 4.3)',              LinearDiscriminantAnalysis())
pipe_qda  = eval_clf('QDA (ESL 4.3)',              QuadraticDiscriminantAnalysis())
pipe_nb   = eval_clf('Naive Bayes (ESL 6.6)',      GaussianNB())
# Modelos clásicos
pipe_lr   = eval_clf('LogisticRegression',          LogisticRegression(max_iter=1000))
pipe_dt   = eval_clf('DecisionTree(d=4)',           DecisionTreeClassifier(max_depth=4))
pipe_rf   = eval_clf('RandomForest(100)',           RandomForestClassifier(n_estimators=100, random_state=42))

## 2 — Curvas ROC y Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
COLORS = ['#4361ee','#f72585','#06d6a0','#ff9f1c','#7209b7']

for (pipe, name), color in zip([
    (pipe_lda, 'LDA'), (pipe_nb, 'Naive Bayes'),
    (pipe_lr, 'Logistic'), (pipe_rf, 'Random Forest')
], COLORS):
    y_prob = pipe.predict_proba(X_test)[:,1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} ({auc:.3f})')

    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    axes[1].plot(rec, prec, color=color, linewidth=2, label=name)

axes[0].plot([0,1],[0,1],'--', color='#aaa', label='Azar')
axes[0].set_xlabel('FPR (1 - Specificity)'); axes[0].set_ylabel('TPR (Recall)')
axes[0].set_title('ROC Curve'); axes[0].legend(fontsize=9)

no_skill = y_test.mean()
axes[1].axhline(no_skill, color='#aaa', linestyle='--', label=f'No skill ({no_skill:.2f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 3 — Calibración de probabilidades (Géron Cap. 3)
Un modelo bien calibrado: si predice 70% de probabilidad → ocurre el 70% de las veces.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# Diagonal perfecta
ax.plot([0,1],[0,1], 'k--', linewidth=1.5, label='Calibración perfecta')

models_cal = [
    (pipe_lr,  'Logistic Regression', '#4361ee'),
    (pipe_rf,  'Random Forest',       '#f72585'),
    (pipe_nb,  'Naive Bayes',         '#06d6a0'),
]

for pipe, name, color in models_cal:
    y_prob = pipe.predict_proba(X_test)[:,1]
    frac_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=10)
    ax.plot(mean_pred, frac_pos, 'o-', color=color, linewidth=2, label=name)

ax.set_xlabel('Probabilidad media predicha')
ax.set_ylabel('Fracción de positivos reales')
ax.set_title('Calibration Curve (Reliability Diagram) — Géron Cap. 3')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('Random Forest tiende a estar mal calibrado (probabilidades extremas)')
print('Logistic Regression generalmente bien calibrado por su formulación probabilística')
print()
print('Fix: usar CalibratedClassifierCV para corregir calibración')

# Calibrar Random Forest
X_prep = preprocessor.fit_transform(X_train)
X_prep_te = preprocessor.transform(X_test)
rf_cal = CalibratedClassifierCV(
    RandomForestClassifier(n_estimators=100, random_state=42),
    method='isotonic', cv=3
)
rf_cal.fit(X_prep, y_train)
y_prob_cal = rf_cal.predict_proba(X_prep_te)[:,1]
frac_cal, mean_cal = calibration_curve(y_test, y_prob_cal, n_bins=10)

fig2, ax2 = plt.subplots(figsize=(6, 5))
ax2.plot([0,1],[0,1], 'k--', linewidth=1.5, label='Perfecto')
y_prob_rf = pipe_rf.predict_proba(X_test)[:,1]
fp_rf, mp_rf = calibration_curve(y_test, y_prob_rf, n_bins=10)
ax2.plot(mp_rf, fp_rf, 'o-', color='#f72585', label='RF sin calibrar')
ax2.plot(mean_cal, frac_cal, 's-', color='#4361ee', label='RF calibrado (isotonic)')
ax2.set_xlabel('Prob. predicha'); ax2.set_ylabel('Fracción positivos')
ax2.set_title('RF antes y después de calibrar')
ax2.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 4 — Clases desbalanceadas (Géron Cap. 3)

In [ ]:
# Simular dataset muy desbalanceado (5% tasa de activación)
np.random.seed(0)
n_imb = 3000
logit_imb = -5 + np.random.randn(n_imb)
y_imb     = (np.random.uniform(0,1,n_imb) < 1/(1+np.exp(-logit_imb))).astype(int)
X_imb     = np.random.randn(n_imb, 4)
X_imb[y_imb==1] += 0.8

X_imb_tr, X_imb_te, y_imb_tr, y_imb_te = train_test_split(X_imb, y_imb, test_size=0.2, random_state=0, stratify=y_imb)
print(f'Tasa positivos: {y_imb.mean():.2%}  — dataset muy desbalanceado')

strategies = [
    ('Sin ajuste',                    RandomForestClassifier(n_estimators=100, random_state=42)),
    ('class_weight=balanced',         RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
    ('class_weight=balanced_subsample', RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample', random_state=42)),
]

print(f'\n{"Estrategia":<35} {"Acc":>6}  {"F1":>6}  {"Recall":>8}  {"AUC":>6}')
print('-'*65)
for name, model in strategies:
    model.fit(X_imb_tr, y_imb_tr)
    yp = model.predict(X_imb_te)
    ypr = model.predict_proba(X_imb_te)[:,1]
    print(f'{name:<35} {accuracy_score(y_imb_te,yp):.3f}  '
          f'{f1_score(y_imb_te,yp):.3f}  '
          f'{recall_score(y_imb_te,yp):.3f}     '
          f'{roc_auc_score(y_imb_te,ypr):.3f}')

## Resumen

| Modelo | Supuesto clave | Cuándo | Referencia |
|---|---|---|---|
| LDA | Gaussianas, $\Sigma$ igual entre clases | Pocas features, distribuciones normales | ESL 4.3 |
| QDA | Gaussianas, $\Sigma_k$ distintas | Más datos, clases con formas distintas | ESL 4.3 |
| Naive Bayes | Features condicionalmente independientes | Texto, datos de alta dimensión | ESL 6.6 |
| LogisticRegression | Límite lineal en log-odds | Default para clasificación binaria | ESL 4.4 |
| RandomForest | Ensemble de árboles | Default robusto para la mayoría | Géron Cap. 7 |
| Calibración | — | Cuando las probabilidades importan (CRM) | Géron Cap. 3 |

**Siguiente:** `05_model_evaluation.ipynb`